
# Notebook 08 — Compare three agents: does curation actually help?

The proof. We run the **same four questions** against the three agents built earlier — all on the same data — and compare accuracy:

| Agent | Built in | What it has |
|---|---|---|
| **Baseline** | 03 | Tables only, minimal instructions |
| **Metric View** | 03b | One governed metric view (pinned joins + KPI formulas) + example `MEASURE()` SQL |
| **Knowledge Store** | 04 | Measures, filters, fields, joins, synonyms, example SQL |

**The questions are KPI-by-dimension** — exactly what both curated approaches are built for. Each one hinges on something a **blank agent** typically gets wrong:

| # | Question | Where a blank agent slips |
|---|----------|---------------------------|
| Q1 | Average **OEE %** — Michigan plants, 2024 | `oee_score` is 0–1 → must `×100`; needs the `quality_metrics_daily → plants` join |
| Q2 | **Scrap rate %** — Texas plants, 2024 | scrap-rate definition (`scrap_count / units_produced` from the daily table) + join |
| Q3 | Average **first-pass yield %** — EV Battery Pack lines, 2024 | FPY `×100` + `quality_metrics_daily → production_lines` product-type join |
| Q4 | **Total units produced** — New York plants, 2024 | `quality_metrics_daily → plants` join + `SUM` |

**What to expect:** the Baseline guesses joins, formulas, and scaling and misses several; the **Metric View** and **Knowledge Store** agents both pin those and should score high. That's the whole point — curation is what makes an agent trustworthy.

**Perimeter note:** the metric view is deterministic *within its scope* (daily line quality). Event-level questions (per-event defect counts, shift analysis) live with the table-based Knowledge Store agent — pair the two in production.

**Before you start:** run **03** (baseline), **03b** (metric view), **04** (knowledge store).

**Compute:** Serverless.

## Load config and connect to the three agents

In [ ]:
%run ./00_workshop_config

## Connect and load the three agent IDs

Create the SDK client and read the Baseline, Metric View, and Knowledge Store ids so we can compare them.

In [ ]:
import re
import time
import json
import requests
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
BASELINE_ID = _cfg.get(CFG_KEY_BASELINE, "")
CURATED_ID = _cfg.get(CFG_KEY_CURATED, "")
METRIC_ID = _cfg.get(CFG_KEY_METRIC_VIEW, "")

if not CURATED_ID:
    raise RuntimeError("genie_space_id (Knowledge Store agent) not found. Run notebook 04 first.")

AGENTS = [(name, sid) for name, sid in [
    ("Baseline", BASELINE_ID),
    ("Metric View", METRIC_ID),
    ("Knowledge Store", CURATED_ID),
] if sid]

print("Comparing:")
for name, sid in AGENTS:
    print(f"  {name:<16} {genie_ui_room_url(sid)}")
if len(AGENTS) < 3:
    print("\nNote: run 03 / 03b / 04 to compare all three. Continuing with what exists.")

## Scoring function

Sends a question to a Genie agent via the Conversation API, waits for the answer, extracts the first number, and compares it to ground-truth SQL within a tolerance.

In [ ]:
def _extract_number(text):
    if text is None:
        return None
    nums = re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    return float(nums[0]) if nums else None


def run_benchmarks(benchmarks, space_id, label):
    """Ask each question, score against ground truth. Returns (results, pass_rate)."""
    print(f"\n{label}")
    results = []
    passes = 0
    for i, b in enumerate(benchmarks, 1):
        gt_val = float(spark.sql(b["gt"]).collect()[0][0])
        genie_val, status = None, "FAIL"
        try:
            start = requests.post(
                f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
                headers=headers, json={"content": b["q"]},
            )
            if start.status_code != 200:
                print(f"  Q{i}: SKIP (start-conversation {start.status_code})")
                results.append((i, b["q"], gt_val, None, "ERROR"))
                continue
            d = start.json()
            cid, mid = d.get("conversation_id"), d.get("message_id")
            for _ in range(40):
                time.sleep(4)
                poll = requests.get(
                    f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}",
                    headers=headers,
                )
                if poll.status_code != 200:
                    continue
                msg = poll.json()
                st = msg.get("status", "")
                if st == "COMPLETED":
                    for att in msg.get("attachments", []):
                        aid = att.get("attachment_id") or att.get("id")
                        if not att.get("query") or not aid:
                            continue
                        qr = requests.get(
                            f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}",
                            headers=headers,
                        )
                        if qr.status_code == 200:
                            arr = qr.json().get("statement_response", {}).get("result", {}).get("data_array", [])
                            if arr and arr[0]:
                                genie_val = _extract_number(arr[0][-1])
                    if genie_val is not None and gt_val != 0:
                        status = "PASS" if abs(genie_val - gt_val) / abs(gt_val) * 100 <= BENCHMARK_TOLERANCE_PCT else "FAIL"
                    elif genie_val is not None and gt_val == 0:
                        status = "PASS" if genie_val == 0 else "FAIL"
                    break
                if st in ("FAILED", "CANCELLED"):
                    break
        except Exception as e:
            print(f"  Q{i}: ERROR ({str(e)[:100]})")
        if status == "PASS":
            passes += 1
        print(f"  Q{i}: {status} (GT={gt_val}, Genie={genie_val})")
        results.append((i, b["q"], gt_val, genie_val, status))
    rate = (passes / len(benchmarks) * 100) if benchmarks else 0
    print(f"  Pass rate: {rate:.0f}% ({passes}/{len(benchmarks)})")
    return results, rate


print("Scoring function ready.")

## The four KPI-by-dimension questions

Ground truth is computed from the base tables (`quality_metrics_daily` joined to `plants`/`production_lines`), so it's independent of any agent's configuration.

In [ ]:
questions = [
    {"q": "What is the average OEE for plants in Michigan in 2024, expressed as a percentage?",
     "gt": f"SELECT CAST(ROUND(AVG(q.oee_score) * 100, 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND YEAR(CAST(q.date AS DATE)) = 2024",
     "short": "Avg OEE % — Michigan, 2024"},
    {"q": "What is the scrap rate for Texas plants in 2024, expressed as a percentage?",
     "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Texas' AND YEAR(CAST(q.date AS DATE)) = 2024",
     "short": "Scrap rate % — Texas, 2024"},
    {"q": "What is the average first-pass yield for EV Battery Pack production lines in 2024, expressed as a percentage?",
     "gt": f"SELECT CAST(ROUND(AVG(q.first_pass_yield) * 100, 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.production_lines pl ON q.production_line_id = pl.line_id WHERE pl.product_type = 'EV Battery Pack' AND YEAR(CAST(q.date AS DATE)) = 2024",
     "short": "Avg FPY % — EV Battery Pack, 2024"},
    {"q": "How many total units were produced across all New York plants in 2024?",
     "gt": f"SELECT CAST(SUM(q.units_produced) AS BIGINT) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'New York' AND YEAR(CAST(q.date AS DATE)) = 2024",
     "short": "Total units — New York, 2024"},
]
print(f"{len(questions)} questions ready.")
print("Ground truth:")
for i, b in enumerate(questions, 1):
    print(f"  Q{i} ({b['short']}): {spark.sql(b['gt']).collect()[0][0]}")

## Run the comparison

Each question goes to every agent. This calls the live Genie API, so it takes a few minutes.

In [ ]:
all_results = {}   # name -> (results, rate)
for name, sid in AGENTS:
    all_results[name] = run_benchmarks(questions, sid, f"=== {name} ===")

## Results side by side

In [ ]:
names = [n for n, _ in AGENTS]
header = f"{'Q#':<4} {'Question':<34}" + "".join(f"{n:<18}" for n in names)
print("=" * len(header))
print("THREE-WAY COMPARISON")
print("=" * len(header))
print(header)
print("-" * len(header))
for idx, hq in enumerate(questions):
    row = f"Q{idx+1:<3} {hq['short']:<34}"
    for n in names:
        res = all_results[n][0][idx]
        row += f"{res[4] + ' (' + str(res[3]) + ')':<18}"
    print(row)
print("-" * len(header))
rate_row = f"{'Pass rate':<39}"
for n in names:
    rate_row += f"{str(int(all_results[n][1])) + '%':<18}"
print(rate_row)
print()
print("Reading the result:")
base = all_results.get("Baseline", (None, None))[1]
for n in ("Metric View", "Knowledge Store"):
    if n in all_results and base is not None:
        r = all_results[n][1]
        verdict = "beats" if r > base else ("matches" if r == base else "trails")
        print(f"  - {n}: {int(r)}%  ({verdict} Baseline {int(base)}%)")
print("  Baseline guesses joins, KPI formulas, and 0-1 vs. percentage scaling and misses several.")
print("  Both curated agents pin those, so they answer reliably — that is what earns user trust.")
print("  (Metric View is deterministic within its governed perimeter; pair it with the")
print("   table-based Knowledge Store agent for event-level questions it can't see.)")

## Save run history

Appends the results to `genie_benchmark_runs` so notebook **11** (monitoring) can chart pass-rate trends.

In [ ]:
ts = datetime.now().isoformat()
sch = StructType([
    StructField("benchmark_id", IntegerType()),
    StructField("question", StringType()),
    StructField("ground_truth", DoubleType()),
    StructField("genie_answer", DoubleType()),
    StructField("status", StringType()),
    StructField("run_timestamp", StringType()),
    StructField("pass_rate", DoubleType()),
    StructField("run_label", StringType()),
])
rows = []
for name, (results, rate) in all_results.items():
    label = "compare_" + name.lower().replace(" ", "_")
    rows += [(r[0], r[1], r[2], r[3], r[4], ts, rate, label) for r in results]

tbl = f"{fqn}.genie_benchmark_runs"
try:
    spark.createDataFrame(rows, sch).write.mode("append").saveAsTable(tbl)
except Exception:
    spark.sql(f"DROP TABLE IF EXISTS {tbl}")
    spark.createDataFrame(rows, sch).write.saveAsTable(tbl)
print(f"Saved {len(rows)} results to {tbl}")

## Validate in the UI (most accurate)

The programmatic scorer compares one number with a tolerance. The Genie **Benchmark** tab compares full result sets and is the most accurate evaluator.

1. Open the **Knowledge Store** and **Metric View** agent links above.
2. Click the **Benchmark** tab → **Start new run**.
3. Review failures; for each, **Review proposed fixes** and accept the knowledge snippets that match your conventions, or **Update ground truth** if Genie's SQL is equivalent.
4. Re-run until green. Target **80%+ before UAT**, higher on clean data.

**Next:** Notebook 09 — Security and governance (column masking).